# KB flow testing — KB match, unknown attack AE, and BENIGN known-test AE
The BENIGN pathway uses the separate audit output and **never inserts BENIGN flows**. Unknown AE-positive flow insertion/retrieval/cleanup uses a temporary KB.

In [8]:
from pathlib import Path
import sys, json, shutil, time
import numpy as np
import pandas as pd
import joblib

ROOT=Path.cwd().resolve()
if ROOT.name.lower()=="kb" and ROOT.parent.name.lower()=="notebooks":
    ROOT=ROOT.parent.parent
elif ROOT.name.lower()=="notebooks":
    ROOT=ROOT.parent
KB_DIR=ROOT/"notebooks"/"kb"
sys.path.insert(0,str(KB_DIR))
from kb_flow import read_partition, build, search, atomic_csv, update_after_ae, load_version20, score_precomputed_flow, prepare_ae_context

SPLITS=ROOT/"data/processed/Data_V3_SSH_Unknown_final/unknown_kb_splits"
KB_DATA=ROOT/"data/processed/knn_kb"
MODELS=ROOT/"saved-models/knn_kb"
RESULTS=ROOT/"outputs/results/knn_kb"
for d in (KB_DATA,MODELS,RESULTS): d.mkdir(parents=True,exist_ok=True)
INITIAL_CSV=KB_DATA/"knowledge_base.csv"
INITIAL_INDEX=MODELS/"initial_kb.joblib"
DEMO_CSV=KB_DATA/"knowledge_base_demo.csv"
DEMO_INDEX=MODELS/"demo_kb.joblib"
for name in ("unknown_initial_kb.parquet","unknown_validation.parquet",
             "unknown_final_test.parquet","unknown_demo_kb_updates.parquet"):
    if not (SPLITS/name).is_file(): raise FileNotFoundError(SPLITS/name)
print("Project:",ROOT)
print("Existing splits:",SPLITS)
print("KB outputs:",KB_DATA)

Project: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection
Existing splits: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\Data_V3_SSH_Unknown_final\unknown_kb_splits
KB outputs: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\knn_kb


In [9]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Load the frozen initial KB without retraining.
ref=pd.read_csv(INITIAL_CSV,low_memory=False)
model=joblib.load(INITIAL_INDEX)
if len(ref)!=model["count"]: raise ValueError("KB/index count mismatch")
if model.get("cutoff") is None: raise ValueError("Missing cutoff; run kb_training.ipynb")
print("Frozen KB:",len(ref),"Cutoff:",model["cutoff"])

Frozen KB: 5203 Cutoff: 0.9453825241925351


In [11]:
# 3. Split the separate demo partition into KB matches and AE candidates.
demo=read_partition(SPLITS/"unknown_demo_kb_updates.parquet")
demo_hits=search(demo,model,ref,k=5)
audit=pd.DataFrame([
    {"demo_row":i,"matched":h["matched"],
     "route":"KB_MATCH" if h["matched"] else "AUTOENCODER",
     "nearest_distance":h["nearest_distance"],
     "nearest_pattern_id":h["neighbors"][0]["pattern_id"],
     "demo_source_row":i}
    for i,h in enumerate(demo_hits)])
mask=audit["matched"].to_numpy(dtype=bool)
tagged=demo.copy()
tagged.insert(0,"demo_row",np.arange(len(demo)))
AUDIT_DIR=KB_DATA/"demo_similarity_audit"
AUDIT_DIR.mkdir(parents=True,exist_ok=True)
tagged.loc[mask].to_parquet(AUDIT_DIR/"kb_demo_matched.parquet",index=False)
tagged.loc[~mask].to_parquet(AUDIT_DIR/"autoencoder_demo_unmatched.parquet",index=False)
audit.to_csv(AUDIT_DIR/"demo_similarity_audit.csv",index=False)
print("=== DEMO SPLIT ===")
print("Total:",len(demo),"KB matches:",int(mask.sum()),"AE candidates:",int((~mask).sum()))
print("Matched:",AUDIT_DIR/"kb_demo_matched.parquet")
print("Unmatched:",AUDIT_DIR/"autoencoder_demo_unmatched.parquet")
display(audit.head(15))

=== DEMO SPLIT ===
Total: 434 KB matches: 420 AE candidates: 14
Matched: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\knn_kb\demo_similarity_audit\kb_demo_matched.parquet
Unmatched: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\knn_kb\demo_similarity_audit\autoencoder_demo_unmatched.parquet


,demo_row,matched,route,nearest_distance,nearest_pattern_id,demo_source_row
0,0,True,KB_MATCH,0.047224,KB_0000086,0
1,1,True,KB_MATCH,0.072170,KB_0001464,1
2,2,True,KB_MATCH,0.113858,KB_0001326,2
3,3,True,KB_MATCH,0.077619,KB_0001901,3
4,4,True,KB_MATCH,0.105062,KB_0000309,4
5,5,True,KB_MATCH,0.285570,KB_0002541,5
6,6,True,KB_MATCH,0.374752,KB_0004624,6
7,7,True,KB_MATCH,0.124801,KB_0002190,7
8,8,True,KB_MATCH,0.031651,KB_0000122,8
9,9,True,KB_MATCH,0.128913,KB_0001168,9


In [12]:
# PATHWAY A: KB match. This uses the frozen initial KB and MUST skip the AE.
matching = audit.loc[audit["matched"]]
if matching.empty:
    print("No matching demo rows are available; KB pathway cannot be demonstrated.")
else:
    kb_row = int(matching.iloc[0]["demo_row"])
    kb_flow = demo.iloc[kb_row].to_dict()
    kb_hit = search(pd.DataFrame([kb_flow]), model, ref, k=5)[0]
    print("\n========== PATHWAY A: KNOWLEDGE BASE ==========")
    print("Incoming flow (full record):")
    print(json.dumps(kb_flow, indent=2, default=str))
    print("Nearest KB pattern:", kb_hit["neighbors"][0]["pattern_id"])
    print("Distance:", kb_hit["nearest_distance"], "Cutoff:", model["cutoff"])
    print("Matched:", kb_hit["matched"])
    assert kb_hit["matched"], "Expected a KB match"
    print("ROUTE: KB MATCH -> AUTOENCODER NOT CALLED -> NO KB UPDATE")



========== PATHWAY A: KNOWLEDGE BASE ==========
Incoming flow (full record):
{
  "id": 22187,
  "Flow ID": "172.16.0.1-192.168.10.50-37792-80-6",
  "Src IP": "172.16.0.1",
  "Src Port": 37792,
  "Dst IP": "192.168.10.50",
  "Dst Port": 80,
  "Protocol": 6,
  "Timestamp": "2017-07-05 13:25:59.331378",
  "Flow Duration": 109686737,
  "Total Fwd Packet": 4,
  "Total Bwd packets": 3,
  "Total Length of Fwd Packet": 362,
  "Total Length of Bwd Packet": 0,
  "Fwd Packet Length Max": 351,
  "Fwd Packet Length Min": 0,
  "Fwd Packet Length Mean": 90.5,
  "Fwd Packet Length Std": 173.7440646468247,
  "Bwd Packet Length Max": 0,
  "Bwd Packet Length Min": 0,
  "Bwd Packet Length Mean": 0.0,
  "Bwd Packet Length Std": 0.0,
  "Flow Bytes/s": 3.300307857640072,
  "Flow Packets/s": 0.0638181077444212,
  "Flow IAT Mean": 18281122.833333336,
  "Flow IAT Std": 44776710.33746833,
  "Flow IAT Max": 109681200,
  "Flow IAT Min": 28,
  "Fwd IAT Total": 109686689,
  "Fwd IAT Mean": 36562229.66666666,
  "Fwd

In [13]:
# ============================================================
# PATHWAY B: KNN MISS -> VERSION20 -> AUTO INSERT
#            -> DIRECT RETRIEVAL -> REMOVE -> VERIFY
# ============================================================

from tempfile import TemporaryDirectory
from pathlib import Path
import pandas as pd
import json
import shutil
import joblib

DEMO_ROW = None  # Set a specific original demo_row if needed

# ------------------------------------------------------------
# STEP 1: SELECT A KB-UNMATCHED DEMO RECORD
# ------------------------------------------------------------

unmatched = audit.loc[~audit["matched"]]

if unmatched.empty:
    print("No unmatched demo records.")
    print("Autoencoder pathway cannot be tested.")

else:
    chosen_row = (
        int(unmatched.iloc[0]["demo_row"])
        if DEMO_ROW is None
        else int(DEMO_ROW)
    )

    if chosen_row not in set(unmatched["demo_row"]):
        raise ValueError(
            f"Demo row {chosen_row} is not unmatched "
            "against the initial Knowledge Base"
        )

    flow = demo.iloc[chosen_row].to_dict()

    report = {
        "demo_row": chosen_row,
        "kb_matched": False,
        "ae_called": False,
        "ae_anomalous": False,
        "inserted": False,
        "retrieved_after_insert": False,
        "after_removal": None,
        "cleaned_up": False
    }

    print("\n========== PATHWAY B: AUTOENCODER ==========")

    print("\n========== INCOMING NETWORK FLOW ==========")
    print(json.dumps(flow, indent=2, default=str))

    # --------------------------------------------------------
    # STEP 2: CREATE TEMPORARY KNOWLEDGE BASE
    # --------------------------------------------------------

    with TemporaryDirectory(prefix="kb_flow_test_") as temporary:

        work = Path(temporary)

        work_csv = work / "knowledge_base_demo.csv"
        work_index = work / "demo_kb.joblib"

        # Copy original KB and KNN index
        shutil.copy2(INITIAL_CSV, work_csv)
        shutil.copy2(INITIAL_INDEX, work_index)

        local_kb = pd.read_csv(
            work_csv,
            low_memory=False
        )

        local_index = joblib.load(work_index)

        print("\n========== TEMPORARY KB CREATED ==========")
        print("Initial KB records:", len(local_kb))

        # ----------------------------------------------------
        # STEP 3: KNN SIMILARITY SEARCH
        # ----------------------------------------------------

        before = search(
            pd.DataFrame([flow]),
            local_index,
            local_kb,
            k=5
        )[0]

        print("\n========== KNN RETRIEVAL ==========")

        print(
            "Nearest KB record:",
            before["neighbors"][0]["pattern_id"]
        )

        print(
            "KNN distance:",
            before["nearest_distance"]
        )

        print(
            "KNN cutoff:",
            local_index["cutoff"]
        )

        print("KB matched:", before["matched"])

        report["before"] = before

        assert not before["matched"], (
            "Expected no match against initial KB"
        )

        print(
            "\nROUTE: KB UNMATCHED -> VERSION20 AUTOENCODER"
        )

        # ----------------------------------------------------
        # STEP 4: GENERATE CHRONOLOGICAL FEATURES
        # ----------------------------------------------------

        prepared, n_prior = prepare_ae_context(
            ROOT,
            flow
        )

        print("\n========== CHRONOLOGICAL FEATURES ==========")

        print("Preceding flows:", n_prior)

        print(
            json.dumps(
                prepared,
                indent=2,
                default=str
            )
        )

        # ----------------------------------------------------
        # STEP 5: VERSION20 AUTOENCODER
        # ----------------------------------------------------

        ae = load_version20(ROOT)

        mse, threshold = score_precomputed_flow(
            prepared,
            ae
        )

        anomalous = bool(mse > threshold)

        report.update({
            "ae_called": True,
            "ae_mse": float(mse),
            "ae_threshold": float(threshold),
            "ae_anomalous": anomalous,
            "prior_flows": int(n_prior)
        })

        print("\n========== VERSION20 AUTOENCODER ==========")

        print("Ensemble reconstruction MSE:", mse)
        print("Global threshold:", threshold)
        print("AE anomalous:", anomalous)

        # ----------------------------------------------------
        # STEP 6: AUTOENCODER DECISION
        # ----------------------------------------------------

        if not anomalous:

            print("\n========== NO KB UPDATE ==========")

            print("ROUTE: AE NEGATIVE")
            print("No anomaly detected.")
            print("Record not inserted into KB.")

        else:

            print("\n========== AUTOMATIC KB INSERTION ==========")

            print(
                "ROUTE: AE POSITIVE -> "
                "AUTOMATIC INSERT INTO TEMPORARY KB"
            )

            # Support current and older kb_flow.py signatures
            import inspect

            parameters = inspect.signature(
                update_after_ae
            ).parameters

            if "approved" in parameters:

                inserted = update_after_ae(
                    flow,
                    mse,
                    threshold,
                    True,
                    work_csv,
                    work_index
                )

            else:

                inserted = update_after_ae(
                    flow,
                    mse,
                    threshold,
                    work_csv,
                    work_index
                )

            inserted_id = inserted["pattern_id"]

            report["inserted"] = True
            report["inserted_pattern_id"] = inserted_id

            print("Inserted pattern ID:", inserted_id)

            # ------------------------------------------------
            # STEP 7: DIRECT RETRIEVAL AFTER INSERTION
            # NO KNN SIMILARITY SEARCH
            # ------------------------------------------------

            new_kb = pd.read_csv(
                work_csv,
                low_memory=False
            )

            retrieved = new_kb[
                "pattern_id"
            ].astype(str).eq(inserted_id).any()

            report["retrieved_after_insert"] = bool(
                retrieved
            )

            print("\n========== DIRECT RECORD RETRIEVAL ==========")

            print(
                "Retrieved after insertion:",
                bool(retrieved)
            )

            assert retrieved, (
                "Inserted record not found in Knowledge Base"
            )

            # ------------------------------------------------
            # STEP 8: REMOVE INSERTED RECORD
            # ------------------------------------------------

            print("\n========== REMOVE TEST RECORD ==========")

            cleaned = new_kb.loc[
                new_kb["pattern_id"].astype(str)
                != inserted_id
            ].copy()

            assert len(cleaned) == len(local_kb), (
                "Cleanup removed unexpected records"
            )

            # Save cleaned Knowledge Base
            atomic_csv(
                cleaned,
                work_csv
            )

            # Restore original KNN index
            shutil.copy2(
                INITIAL_INDEX,
                work_index
            )

            print("Removed pattern ID:", inserted_id)

            # ------------------------------------------------
            # STEP 9: DIRECT LOOKUP AFTER REMOVAL
            # NO KNN SIMILARITY SEARCH
            # ------------------------------------------------

            record_lookup = cleaned.loc[
                cleaned["pattern_id"].astype(str)
                == inserted_id
            ]

            if record_lookup.empty:
                retrieved_after_removal = None
            else:
                retrieved_after_removal = (
                    record_lookup.iloc[0].to_dict()
                )

            report["after_removal"] = (
                retrieved_after_removal
            )

            print("\n========== VERIFY RECORD REMOVAL ==========")

            print(
                "Record after removal:",
                retrieved_after_removal
            )

            assert retrieved_after_removal is None, (
                "Inserted record still exists in KB"
            )

        report["cleaned_up"] = True

    # --------------------------------------------------------
    # STEP 10: FINAL RESULTS
    # --------------------------------------------------------

    print("\n========== TEMPORARY KB CLEANUP ==========")

    print("Temporary Knowledge Base deleted.")
    print("Original Knowledge Base unchanged.")

    print("\n========== FINAL AE PATHWAY RESULT ==========")

    print(
        json.dumps(
            report,
            indent=2,
            default=str
        )
    )

    # Save test results
    RESULTS.mkdir(
        parents=True,
        exist_ok=True
    )

    result_file = (
        RESULTS / "demo_flow_test_result.json"
    )

    result_file.write_text(
        json.dumps(
            report,
            indent=2,
            default=str
        ),
        encoding="utf-8"
    )

    print("\nResults saved to:", result_file)


========== PATHWAY B: AUTOENCODER ==========

========== INCOMING NETWORK FLOW ==========
{
  "id": 234596,
  "Flow ID": "172.16.0.1-192.168.10.50-46592-22-6",
  "Src IP": "172.16.0.1",
  "Src Port": 46592,
  "Dst IP": "192.168.10.50",
  "Dst Port": 22,
  "Protocol": 6,
  "Timestamp": "2017-07-04 17:12:07.991505",
  "Flow Duration": 12039789,
  "Total Fwd Packet": 25,
  "Total Bwd packets": 35,
  "Total Length of Fwd Packet": 2072,
  "Total Length of Bwd Packet": 2825,
  "Fwd Packet Length Max": 640,
  "Fwd Packet Length Min": 0,
  "Fwd Packet Length Mean": 82.87999999999998,
  "Fwd Packet Length Std": 131.75492400665715,
  "Bwd Packet Length Max": 976,
  "Bwd Packet Length Min": 0,
  "Bwd Packet Length Mean": 80.71428571428574,
  "Bwd Packet Length Std": 211.2661509911653,
  "Flow Bytes/s": 406.73470274271415,
  "Flow Packets/s": 4.983476039322616,
  "Flow IAT Mean": 204064.22033898305,
  "Flow IAT Std": 578650.9760199406,
  "Flow IAT Max": 2256127,
  "Flow IAT Min": 3,
  "Fwd IAT To

In [14]:
# PATHWAY C: BENIGN from known_test.parquet -> frozen KNN -> Version20 on KB miss.
# No BENIGN record is inserted into the KB, even if AE flags it anomalous.
AE_DIR=ROOT/"notebooks"/"autoencoder"
if str(AE_DIR) not in sys.path: sys.path.insert(0,str(AE_DIR))
from version20_common import add_behavioural_features
KNOWN_FILE=ROOT/"data/processed/Data_V3_SSH_Unknown_final/known_test.parquet"
# BENIGN testing is independent of the attack-only demo audit.
BENIGN_SAMPLE_SIZE = 20
BENIGN_RANDOM_SEED = 42
known=pd.read_parquet(KNOWN_FILE)
known.columns=known.columns.astype(str).str.strip()
if "Label" not in known: raise KeyError("known_test missing Label")
valid=known["Label"].astype(str).str.strip().str.upper().eq("BENIGN")
if "is_attempted" in known:
    valid &= ~known["is_attempted"].astype(str).str.strip().str.lower().isin(["true","1","yes","t"])
positions=np.flatnonzero(valid.to_numpy())
if not len(positions): raise ValueError("No non-attempted BENIGN rows in known_test")
# Find BENIGN flows that miss the initial KB, so Version20 actually runs.
benign_candidates=known.iloc[positions].reset_index(drop=True)
benign_hits=search(benign_candidates,model,ref,k=1)
misses=np.array([positions[i] for i,h in enumerate(benign_hits) if not h["matched"]],dtype=int)
rng=np.random.default_rng(BENIGN_RANDOM_SEED)
selected=rng.choice(misses,size=min(BENIGN_SAMPLE_SIZE,len(misses)),replace=False)
benign_demo=known.iloc[selected].copy()
benign_demo.insert(0,"known_test_row",selected)
print("BENIGN total:",len(positions),"KB-unmatched:",len(misses),"selected for AE:",len(selected))
if benign_demo.empty:
    print("No KB-unmatched BENIGN flows available; cannot test BENIGN AE pathway.")
else:
    benign_reports=[]
    ae_loaded=None
    times=pd.to_datetime(known['Timestamp'],errors='coerce')
    if times.isna().any(): raise ValueError('Invalid timestamps in known_test')
    for _,row in benign_demo.iterrows():
        original_idx=int(row['known_test_row'])
        flow=known.iloc[original_idx].to_dict()
        if str(flow.get('Label','')).strip().upper()!='BENIGN':
            raise ValueError('Selected record is not BENIGN')
        lookup=search(pd.DataFrame([flow]),model,ref,k=5)[0]
        entry={'known_test_row':original_idx,'kb_matched':bool(lookup['matched']),
               'nearest_distance':float(lookup['nearest_distance']),
               'ae_called':False,'kb_updated':False}
        print('\n========== BENIGN FLOW',original_idx,'==========')
        print('Incoming flow:',json.dumps(flow,indent=2,default=str))
        print('KNN match:',lookup['matched'],'distance:',lookup['nearest_distance'])
        if lookup['matched']:
            entry['outcome']='KB_MATCH_AE_SKIPPED'
            print('ROUTE: KB MATCH -> AE skipped (potentially inappropriate for BENIGN)')
        else:
            # Original source ordering is the tie-break for equal timestamps.
            prior=(times<times.iloc[original_idx]) | ((times==times.iloc[original_idx]) & (np.arange(len(known))<original_idx))
            context=pd.concat([known.loc[prior],known.iloc[[original_idx]]],ignore_index=True)
            prepared=add_behavioural_features(context).iloc[-1].to_dict()
            if ae_loaded is None: ae_loaded=load_version20(ROOT)
            mse,threshold=score_precomputed_flow(prepared,ae_loaded)
            anomalous=bool(mse>threshold)
            entry.update(ae_called=True,ae_mse=float(mse),ae_threshold=float(threshold),
                         ae_anomalous=anomalous,
                         outcome='FALSE_POSITIVE_ANOMALY' if anomalous else 'NORMAL_NO_ANOMALY')
            print('ROUTE: AUTOENCODER')
            print('Chronological preceding known-test flows:',int(prior.sum()))
            print('Ensemble MSE:',mse,'Global threshold:',threshold)
            print('OUTCOME:',entry['outcome'],'| NO KB INSERTION')
        benign_reports.append(entry)
    benign_report=pd.DataFrame(benign_reports)
    OUT=ROOT/'data/processed/knn_kb/demo_similarity_audit'
    OUT.mkdir(parents=True,exist_ok=True)
    benign_report.to_csv(OUT/'benign_demo_results.csv',index=False)
    print('\n========== BENIGN PATHWAY SUMMARY ==========')
    print(benign_report['outcome'].value_counts().to_string())
    print('Saved:',OUT/'benign_demo_results.csv')
    print('Note: chronological history uses known_test only, which may differ from original Version20 evaluation.')


BENIGN total: 213767 KB-unmatched: 213767 selected for AE: 20

========== BENIGN FLOW 211335 ==========
Incoming flow: {
  "id": 231730,
  "Flow ID": "192.168.10.17-42.123.102.84-58456-443-6",
  "Src IP": "192.168.10.17",
  "Src Port": 58456,
  "Dst IP": "42.123.102.84",
  "Dst Port": 443,
  "Protocol": 6,
  "Timestamp": "2017-07-07 17:08:08.334130",
  "Flow Duration": 1191005,
  "Total Fwd Packet": 4,
  "Total Bwd packets": 2,
  "Total Length of Fwd Packet": 31,
  "Total Length of Bwd Packet": 0,
  "Fwd Packet Length Max": 31,
  "Fwd Packet Length Min": 0,
  "Fwd Packet Length Mean": 7.749999999999999,
  "Fwd Packet Length Std": 15.5,
  "Bwd Packet Length Max": 0,
  "Bwd Packet Length Min": 0,
  "Bwd Packet Length Mean": 0.0,
  "Bwd Packet Length Std": 0.0,
  "Flow Bytes/s": 26.02843816776588,
  "Flow Packets/s": 5.037762226019202,
  "Flow IAT Mean": 238201.0,
  "Flow IAT Std": 271903.61885142315,
  "Flow IAT Max": 663611,
  "Flow IAT Min": 3,
  "Fwd IAT Total": 1191005,
  "Fwd IAT Me